In [11]:
# %% CELL 1 — install + imports
!pip install -q anthropic

import json
import time
import concurrent.futures
import pandas as pd
from kaggle_secrets import UserSecretsClient
import anthropic

client = anthropic.Anthropic(
    api_key=UserSecretsClient().get_secret("ANTHROPIC_API_KEY")
)
MODEL = "claude-sonnet-4-5"

TARGETS = ["ACL", "MCL", "Medial Meniscus", "Lateral Meniscus", "Medial OA",
           "Lateral OA", "PF OA", "Effusion", "Synovitis", "Baker's",
           "Contusion", "Fracture"]

# "affirmed"/"negated"/"absent" are global - the FP/FN breakdown from the
# first validation run showed no systematic problem with those three.
# "hedged" needed to vary per label: most labels resolve hedge/mild/trace
# language as NEGATIVE in gold (matches the original regex-labeler
# calibration), so hedged sits below 0.5 for those. Synovitis is the
# exception - gold calls inferred/hedged synovitis positive more often than
# not, so its hedged value stays above 0.5.
GLOBAL_TARGET = {"affirmed": 1.0, "negated": 0.1, "absent": 0.28}
HEDGED_BY_LABEL = {
    "ACL": 0.35,
    "MCL": 0.35,
    "Medial Meniscus": 0.35,
    "Lateral Meniscus": 0.40,
    "Medial OA": 0.35,
    "Lateral OA": 0.35,
    "PF OA": 0.45,
    "Effusion": 0.42,
    "Synovitis": 0.55,
    "Baker's": 0.35,
    "Contusion": 0.30,
    "Fracture": 0.45,
}
VALID_STATUSES = {"affirmed", "hedged", "negated", "absent"}

def status_to_target(finding, status):
    if status == "hedged":
        return HEDGED_BY_LABEL[finding]
    return GLOBAL_TARGET[status]

In [12]:
# %% CELL 2 — the prompt
SYSTEM_PROMPT = """You are extracting structured findings from a knee MRI radiology report for a machine learning dataset. The report may be in any language (English, Spanish, French, Dutch, German, Turkish, Croatian, Greek, Bulgarian, or others) — read it in its original language.

IMPORTANT — negation direction varies by language. Do not assume negation words always come before the finding. In particular, Turkish negates AFTER the term (e.g. "efüzyon izlenmedi" = "effusion [was] not observed" — the negation "izlenmedi" follows "efüzyon"). Reason about the actual grammar of whichever language the report is in rather than pattern-matching a fixed word order, and check the sentence carefully in both directions before deciding a finding is negated.

For each of these 12 findings, classify its mention status as exactly one of:
- "affirmed": the report clearly and confidently states this finding is present.
- "hedged": the finding is mentioned but with uncertain, mild, trace, borderline, or inferred language — not confidently affirmed, but not absent either.
- "negated": the report explicitly states this finding is absent, ruled out, or normal for this specific structure.
- "absent": the finding is never mentioned anywhere in the report — no affirmation, no hedge, no explicit negation. This is different from "negated": silence is not the same as an explicit "no."

Per-finding criteria for what counts as "affirmed" vs "hedged" vs "negated":

- ACL: "affirmed" = clear injury (grade 2-3 sprain, partial or complete tear). "hedged" = a bare grade-1 sprain, "interstitial" signal change alone, or a contour-normal ligament with only mild signal change, UNLESS the impression/conclusion emphasizes a real injury (then "affirmed"). "negated" = report states the ACL is intact/normal.
- MCL: same standard as ACL.
- Medial Meniscus: "affirmed" = any stated tear (any grade). "hedged" = degenerative signal alone without a stated tear. "negated" = explicitly intact.
- Lateral Meniscus: same standard as medial.
- Medial OA: "affirmed" = real cartilage loss, chondropathy grade 3+, osteophytes, or an explicit "osteoarthritis"/"arthrosis" statement covering the medial femorotibial compartment. "hedged" = mild/low-grade cartilage fissuring or thinning alone. "negated" = explicitly normal cartilage/no arthrosis in this compartment.
- Lateral OA: same standard as Medial OA, lateral femorotibial compartment.
- PF OA: same standard as Medial OA, patellofemoral compartment (patella/trochlea).
- Effusion: "affirmed" = a qualifier of "moderate" or stronger (moderate, large, significant, tense, hemarthrosis), OR a "small"/"mild" effusion that is also restated in the impression/conclusion. "hedged" = a "trace," "minimal," or "physiologic" amount, or a "small"/"mild" effusion mentioned only once in the body findings and not echoed in the impression. "negated" = explicitly no effusion.
- Synovitis: "affirmed" = explicit synovial thickening/proliferation/inflammation stated as synovitis. "hedged" = synovitis not explicitly mentioned, but the report describes at least a moderate (or stronger) effusion — synovial thickening is frequently visible on the actual images and read into the diagnosis even when the written report doesn't separately call it out, and moderate-or-larger effusion is the most reliable textual proxy for it. Also "hedged" for general "arthritis," Hoffa fat-pad inflammation ("hoffitis"), or bursitis mentioned alone without a direct synovitis statement. "negated" = explicitly no synovitis/synovial thickening.
- Baker's: "affirmed" = popliteal (Baker's) cyst with any real measured size. "hedged" = a "trace" or barely-perceptible one with no size given. "negated" = explicitly no popliteal cyst.
- Contusion: "affirmed" = bone marrow edema/contusion OR muscle contusion, ONLY when the report uses explicit trauma/contusion language (contusion, bone bruise, bruise, impaction, kissing contusion, acute) or is in a clear acute-injury context (e.g. accompanies a fracture or a described traumatic mechanism). "hedged" = marrow edema described as reactive, subchondral under a cartilage defect, or part of a degenerative/osteoarthritic process — this usually belongs to OA, not trauma, so don't affirm it here even if extensive; also use "hedged" when genuinely uncertain whether edema is traumatic or degenerative. "negated" = explicitly no contusion/bone bruise.
- Fracture: "affirmed" = any fracture stated in explicit fracture language (including stress/insufficiency/subchondral fractures and bony avulsions), OR a description of the imaging appearance of a fracture even when the word "fracture" is never used — e.g. a discrete linear/curvilinear marrow edema pattern crossing or reaching the cortex, a cortical step-off, discontinuity, or disruption, a visible lucent fracture line, or an impaction/depression pattern consistent with an acute fracture. "hedged" = marrow edema or a subtle cortical irregularity that could represent an early/occult fracture but is not clearly described as a discrete line or cortical break, or fracture language paired with hedging ("possible fracture," "cannot exclude a fracture"). Do not count surgical "microfracture" procedures or non-clinical mentions like "fracture risk" — treat those as "absent," not "affirmed." "negated" = explicitly no fracture.

Read the whole report, including the impression/conclusion, before deciding — the conclusion is usually the more reliable summary of what the radiologist actually considers clinically significant, and can override a more tentative mention in the body of the findings.

Respond with ONLY a JSON object, no other text, mapping each finding name to one of "affirmed", "hedged", "negated", "absent", in this exact form:
{"ACL": "absent", "MCL": "absent", "Medial Meniscus": "absent", "Lateral Meniscus": "absent", "Medial OA": "absent", "Lateral OA": "absent", "PF OA": "absent", "Effusion": "absent", "Synovitis": "absent", "Baker's": "absent", "Contusion": "absent", "Fracture": "absent"}
"""

def label_report(report_text: str) -> dict:
    resp = client.messages.create(
        model=MODEL,
        max_tokens=300,
        system=SYSTEM_PROMPT,
        messages=[{"role": "user", "content": report_text}],
    )
    text = resp.content[0].text.strip()
    if text.startswith("```"):
        text = text.strip("`")
        text = text[text.find("{"):text.rfind("}") + 1]
    statuses = json.loads(text)
    bad = {t: s for t, s in statuses.items() if s not in VALID_STATUSES}
    if bad:
        raise ValueError(f"invalid status value(s): {bad}")
    return {t: status_to_target(t, statuses[t]) for t in TARGETS}

In [13]:
# %% CELL 3 — validate against the 58 gold-labeled studies FIRST
train = pd.read_csv("/kaggle/input/competitions/rsna-knee-abnormality-detection/train.csv")
gold = train.dropna(subset=TARGETS)
print(f"gold-labeled studies: {len(gold)}")

def validate_one(row):
    try:
        pred = label_report(row["Report"])
        true = {t: int(row[t]) for t in TARGETS}
        return row["StudyInstanceUID"], pred, true, None
    except Exception as e:
        return row["StudyInstanceUID"], None, None, str(e)

results = []
with concurrent.futures.ThreadPoolExecutor(max_workers=8) as ex:
    for r in ex.map(validate_one, [row for _, row in gold.iterrows()]):
        results.append(r)

errors = [r for r in results if r[3] is not None]
print(f"{len(errors)} calls failed")

# per-label accuracy + FP/FN breakdown (soft prediction binarized at 0.5 vs gold's hard 0/1)
correct = {t: 0 for t in TARGETS}
total = {t: 0 for t in TARGETS}
false_pos = {t: 0 for t in TARGETS}
false_neg = {t: 0 for t in TARGETS}
for uid, pred, true, err in results:
    if pred is None:
        continue
    for t in TARGETS:
        total[t] += 1
        pred_hard = int(pred[t] >= 0.5)
        if pred_hard == true[t]:
            correct[t] += 1
        elif pred_hard == 1 and true[t] == 0:
            false_pos[t] += 1
        elif pred_hard == 0 and true[t] == 1:
            false_neg[t] += 1

for t in TARGETS:
    if total[t]:
        print(f"{t:20s} {correct[t]}/{total[t]}  {correct[t]/total[t]:.1%}  "
              f"(FP={false_pos[t]}, FN={false_neg[t]})")

overall = sum(correct.values()) / max(sum(total.values()), 1)
print(f"\noverall label accuracy (binarized @0.5): {overall:.1%}")

gold-labeled studies: 58
0 calls failed
ACL                  55/58  94.8%  (FP=3, FN=0)
MCL                  55/58  94.8%  (FP=3, FN=0)
Medial Meniscus      51/58  87.9%  (FP=4, FN=3)
Lateral Meniscus     52/58  89.7%  (FP=2, FN=4)
Medial OA            52/58  89.7%  (FP=5, FN=1)
Lateral OA           51/58  87.9%  (FP=5, FN=2)
PF OA                53/58  91.4%  (FP=0, FN=5)
Effusion             46/58  79.3%  (FP=1, FN=11)
Synovitis            40/58  69.0%  (FP=9, FN=9)
Baker's              54/58  93.1%  (FP=3, FN=1)
Contusion            47/58  81.0%  (FP=9, FN=2)
Fracture             46/58  79.3%  (FP=7, FN=5)

overall label accuracy (binarized @0.5): 86.5%


In [14]:
# %% CELL 4 — once validation looks good, label the remaining unlabeled reports
unlabeled = train[train[TARGETS].isna().any(axis=1)]
print(f"unlabeled studies to process: {len(unlabeled)}")

def label_one(row):
    try:
        pred = label_report(row["Report"])
        return row["StudyInstanceUID"], pred, None
    except Exception as e:
        return row["StudyInstanceUID"], None, str(e)

silver_rows = []
failed = []
CHUNK = 200
rows = list(unlabeled.iterrows())

for c0 in range(0, len(rows), CHUNK):
    block = rows[c0:c0 + CHUNK]
    with concurrent.futures.ThreadPoolExecutor(max_workers=8) as ex:
        for uid, pred, err in ex.map(label_one, [r for _, r in block]):
            if pred is not None:
                silver_rows.append({"StudyInstanceUID": uid, **pred})
            else:
                failed.append((uid, err))
    print(f"{min(c0 + CHUNK, len(rows))}/{len(rows)} done, {len(failed)} failed so far")
    pd.DataFrame(silver_rows).to_csv("/kaggle/working/report_labels_soft_llm.csv", index=False)

print(f"done. {len(silver_rows)} labeled, {len(failed)} failed")
if failed:
    pd.DataFrame(failed, columns=["StudyInstanceUID", "error"]).to_csv(
        "/kaggle/working/report_labels_failed.csv", index=False)

unlabeled studies to process: 4349
200/4349 done, 0 failed so far
400/4349 done, 0 failed so far
600/4349 done, 0 failed so far
800/4349 done, 1 failed so far
1000/4349 done, 1 failed so far
1200/4349 done, 1 failed so far
1400/4349 done, 1 failed so far
1600/4349 done, 1 failed so far
1800/4349 done, 1 failed so far
2000/4349 done, 1 failed so far
2200/4349 done, 1 failed so far
2400/4349 done, 1 failed so far
2600/4349 done, 1 failed so far
2800/4349 done, 1 failed so far
3000/4349 done, 1 failed so far
3200/4349 done, 1 failed so far
3400/4349 done, 1 failed so far
3600/4349 done, 1 failed so far
3800/4349 done, 1 failed so far
4000/4349 done, 1 failed so far
4200/4349 done, 1 failed so far
4349/4349 done, 1 failed so far
done. 4348 labeled, 1 failed


In [15]:
# %% CELL 5 — merge gold + silver into one labels table, with a source flag
silver = pd.read_csv("/kaggle/working/report_labels_soft_llm.csv")
silver["label_source"] = "silver_llm_soft"

gold_out = gold[["StudyInstanceUID"] + TARGETS].copy()
gold_out["label_source"] = "gold"

full_labels = pd.concat([gold_out, silver[["StudyInstanceUID"] + TARGETS + ["label_source"]]],
                        ignore_index=True)
full_labels.to_csv("/kaggle/working/report_labels_v2.csv", index=False)
print(full_labels["label_source"].value_counts())
print(f"total labeled studies: {len(full_labels)} / {len(train)}")

label_source
silver_llm_soft    4348
gold                 58
Name: count, dtype: int64
total labeled studies: 4406 / 4407


In [16]:
# %% CELL 6 — get a direct download link for the final CSV
from IPython.display import FileLink
FileLink("report_labels_v2.csv")

/kaggle/working/report_labels_v2.csv